# Kharcha parser: LoRA fine-tune (Kaggle / Colab, free GPU)

Everything below calls `kharcha_ml` so runs are reproducible from the package (PROJECT_SPEC §15.4).
`gold_test` is never trained on. Enable a GPU (T4/P100) before running.

In [ ]:
!git clone https://github.com/AdepuTriveni/kharcha.git && cd kharcha/ml && pip -q install uv && uv sync --extra train

## 1. Dataset
Synthetic data is generated here. Real, consented, reviewed labels are uploaded as a private Kaggle dataset (never committed) and passed to `build`.

In [ ]:
%cd kharcha/ml
!uv run kharcha-dataset synth --n 8000 --out data/synthetic/synthetic.jsonl
# !uv run kharcha-dataset build data/synthetic/synthetic.jsonl /kaggle/input/kharcha-labels/distilled.jsonl --reviewed /kaggle/input/kharcha-labels/labels.jsonl --out data/processed
!uv run kharcha-dataset build data/synthetic/synthetic.jsonl --reviewed data/samples/parsing_sample.jsonl --out data/processed
!cat data/DATASET_CARD.md | head -20

## 2. Zero-shot base comparison and LoRA training
Try 2–3 bases on `val` first (e.g. Qwen2.5-0.5B/1.5B-Instruct, Gemma-2-2B-it); train the best.

In [ ]:
!uv run python -m kharcha_ml.finetune.train --base Qwen/Qwen2.5-0.5B-Instruct --data data/processed --out runs/qwen05-lora --epochs 3
!cat runs/qwen05-lora/metrics.json

## 3. Export to GGUF (Q4_K_M) for Ollama and the phone

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp && cd llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j 4
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!uv run kharcha-export --adapter runs/qwen05-lora/adapter --base Qwen/Qwen2.5-0.5B-Instruct --llama-cpp llama.cpp --version v1 --out exports
!cat exports/v1/manifest.json

## 4. Download `exports/v1/` and on your machine:
```
ollama create kharcha-parser:v1 -f exports/v1/Modelfile
ollama create kharcha-parser:v1-f16 -f exports/v1/Modelfile.f16
cd ml && uv run kharcha-ml-eval data/processed/gold_test.jsonl --variant base=qwen2.5:0.5b --variant fp16=kharcha-parser:v1-f16 --variant q4=kharcha-parser:v1 --variant teacher=qwen2.5:7b --rules-report ../backend/reports/parsing/<rules>.json --out reports/v1
cd ../backend && uv run kharcha-admin register-model ../ml/exports/v1/manifest.json --eval ../ml/reports/v1/eval_report.json
```